# HAC Estimators Monte Carlo Simulation

This notebook:
1. Simulates three different processes:
   - a) Regression model that meets GM assumptions
   - b) Regression model with heteroskedastic errors
   - c) Regression model with AC errors
2. Performs parameter inference using normal, White and NW standard errors
3. Evaluates whether tests have correct size

Converted from MATLAB/R implementation.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from olshac import olshac
from armasim import armasim
import time

In [ ]:
# Clear output
print("\n" + "=" * 70)
print("HAC Estimators Monte Carlo Simulation")
print("=" * 70)

## Set Random Seed and Parameters

In [ ]:
# Set the seed for reproducibility
np.random.seed(123467)

# Parameters for DGP
# y = x*b + u
n = 200                    # length of the simulated series
nsim = 10000               # number of simulations
x = np.column_stack([np.ones(n), np.random.randn(n, 2)])  # constant and two random series
b = np.array([0.5, 1.0, -0.5])  # true parameter vector
sig = 0.1                  # standard deviation of error terms

print(f"Number of observations (n): {n}")
print(f"Number of simulations: {nsim}")
print(f"True parameters: {b}")
print(f"Standard deviation of errors: {sig}")

## ARMA Error Parameters

In [ ]:
# Parameters for the ARMA error terms
p = 1          # AR order
q = 1          # MA order
alpha = 0.0    # constant
phi = 0.9      # AR parameter
theta = 0.7    # MA parameter
sigar = 0.1    # standard error of innovation term

print(f"\nARMA({p},{q}) Error Structure:")
print(f"AR parameter (phi): {phi}")
print(f"MA parameter (theta): {theta}")
print(f"Innovation std dev: {sigar}")

## Run Monte Carlo Simulation

In [ ]:
# Initialize matrix to save t-statistics
# Col 0: OLS se, Col 1: White, Col 2: NW
# Only save results for b[1] (second coefficient)
tsave = np.zeros((nsim, 3))

print("\nRunning Monte Carlo simulation...")
start_time = time.time()

# Simulate model with GM, heteroskedastic, or autocorrelated errors
for i in range(nsim):
    # Uncomment one of the following error specifications:
    # gmerr = np.random.randn(n) * sig                           # GM errors
    # hserr = np.concatenate([np.random.randn(n//2) * sig,      # heteroskedastic errors
    #                         np.random.randn(n//2) * (sig/5)])
    arerr = armasim(n, p, q, alpha, [phi], [theta], sigar)      # AR errors
    
    y = x @ b + arerr                      # DGP with selected errors
    results = olshac(y, x, output=0)       # estimate and save results
    bmb0 = results['b'][1] - b[1]          # difference of estimated value from true value
    tsave[i, :] = [bmb0 / results['bse'][1],
                   bmb0 / results['wh_bse'][1],
                   bmb0 / results['nw_bse'][1]]
    
    # Progress indicator
    if (i + 1) % 1000 == 0:
        print(f"Progress: {i+1}/{nsim} ({(i+1)/nsim*100:.1f}%)")

elapsed_time = time.time() - start_time
print(f"\nSimulation completed in {elapsed_time:.2f} seconds")
print(f"Average time per iteration: {elapsed_time/nsim*1000:.2f} ms")

## Calculate Rejection Rates

In [ ]:
# Critical values
cv5 = stats.norm.ppf(0.975)              # 2-tailed 5% cv, Normal
cv1 = stats.norm.ppf(0.995)              # 2-tailed 1% cv, Normal
cv5t = stats.t.ppf(0.975, df=n-len(b))  # 2-tailed 5% cv, t-Distribution
cv1t = stats.t.ppf(0.995, df=n-len(b))  # 2-tailed 1% cv, t-Distribution

print(f"\nCritical Values:")
print(f"5% level (Normal): ±{cv5:.4f}")
print(f"1% level (Normal): ±{cv1:.4f}")
print(f"5% level (t-dist): ±{cv5t:.4f}")
print(f"1% level (t-dist): ±{cv1t:.4f}")

# Count rejections (1 if test stat exceed cv, 0 otherwise)
count5 = np.column_stack([np.abs(tsave[:, 0]) > cv5t,
                          np.abs(tsave[:, 1:3]) > cv5])
count1 = np.column_stack([np.abs(tsave[:, 0]) > cv1t,
                          np.abs(tsave[:, 1:3]) > cv1])

# Calculate proportion of rejections of true H0
prop5 = np.sum(count5, axis=0) / nsim  # proportion at 5%
prop1 = np.sum(count1, axis=0) / nsim  # proportion at 1%

## Display Results

The rejection rates represent the proportion of times we reject the true null hypothesis. Under correct specification, these should be close to the nominal significance levels (5% and 1%).

In [ ]:
print("\n" + "=" * 70)
print("Rejections for standard OLS, White and NW based t-tests")
print("=" * 70)
print("   OLS       White       NW")
print("at 5%")
print(f"{prop5[0]:7.4f}   {prop5[1]:7.4f}   {prop5[2]:7.4f}")
print("at 1%")
print(f"{prop1[0]:7.4f}   {prop1[1]:7.4f}   {prop1[2]:7.4f}")
print("=" * 70)

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of t-statistics
ax = axes[0, 0]
for i, label in enumerate(['OLS', 'White', 'Newey-West']):
    ax.hist(tsave[:, i], bins=50, alpha=0.5, label=label, density=True)
# Add standard normal for comparison
x_norm = np.linspace(-4, 4, 100)
ax.plot(x_norm, stats.norm.pdf(x_norm), 'r--', linewidth=2, label='N(0,1)')
ax.axvline(x=cv5t, color='black', linestyle='--', alpha=0.5)
ax.axvline(x=-cv5t, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('t-statistic')
ax.set_ylabel('Density')
ax.set_title('Distribution of t-statistics')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Rejection rates comparison
ax = axes[0, 1]
x_pos = np.arange(3)
width = 0.35
ax.bar(x_pos - width/2, prop5, width, label='5% level', alpha=0.8)
ax.bar(x_pos + width/2, prop1, width, label='1% level', alpha=0.8)
ax.axhline(y=0.05, color='r', linestyle='--', label='Nominal 5%')
ax.axhline(y=0.01, color='b', linestyle='--', label='Nominal 1%')
ax.set_ylabel('Rejection Rate')
ax.set_title('Rejection Rates by Estimator')
ax.set_xticks(x_pos)
ax.set_xticklabels(['OLS', 'White', 'Newey-West'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 3. Q-Q plot for OLS
ax = axes[1, 0]
stats.probplot(tsave[:, 0], dist="norm", plot=ax)
ax.set_title('Q-Q Plot: OLS t-statistics vs Normal')
ax.grid(True, alpha=0.3)

# 4. Time series of t-statistics (first 100 simulations)
ax = axes[1, 1]
n_plot = 100
for i, label in enumerate(['OLS', 'White', 'Newey-West']):
    ax.plot(tsave[:n_plot, i], alpha=0.7, label=label)
ax.axhline(y=cv5t, color='r', linestyle='--', alpha=0.5, label='5% CV')
ax.axhline(y=-cv5t, color='r', linestyle='--', alpha=0.5)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('Simulation')
ax.set_ylabel('t-statistic')
ax.set_title(f'First {n_plot} t-statistics')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Calculate summary statistics for t-statistics
summary_stats = pd.DataFrame({
    'OLS': [
        np.mean(tsave[:, 0]),
        np.std(tsave[:, 0]),
        np.percentile(tsave[:, 0], 2.5),
        np.percentile(tsave[:, 0], 97.5)
    ],
    'White': [
        np.mean(tsave[:, 1]),
        np.std(tsave[:, 1]),
        np.percentile(tsave[:, 1], 2.5),
        np.percentile(tsave[:, 1], 97.5)
    ],
    'Newey-West': [
        np.mean(tsave[:, 2]),
        np.std(tsave[:, 2]),
        np.percentile(tsave[:, 2], 2.5),
        np.percentile(tsave[:, 2], 97.5)
    ]
}, index=['Mean', 'Std Dev', '2.5th percentile', '97.5th percentile'])

print("\nSummary Statistics of t-statistics:")
print(summary_stats.to_string())
print("\nTheoretical values for N(0,1):")
print(f"Mean: 0.0000")
print(f"Std Dev: 1.0000")
print(f"2.5th percentile: {stats.norm.ppf(0.025):.4f}")
print(f"97.5th percentile: {stats.norm.ppf(0.975):.4f}")